# Sensor Network Reliability & Spatial Heat Gradient Recipe

This recipe combines 3 `algebrax` tools to evaluate IoT wireless sensor networks:

1. **Max-Product Path Link Reliability** (`algebrax.semiring.ViterbiSemiring` & `algebrax.matrix.core.power`):
   Calculates maximum path probability $P_{\max}(u \to v)$ over lossy multi-hop links using Viterbi semiring $([0, 1], \max, \times)$.
2. **Spatial RBF Gaussian Affinity & Sparsity** (`algebrax.analysis.gaussian_kernel` & `algebrax.metrics.sparsity`):
   Transforms distance matrices into spatial similarity $K_{ij} = \exp(-d_{ij}^2 / (2 \sigma^2))$ and audits connectivity density.
3. **Discrete Thermal Field Gradients** (`algebrax.analysis.gradient`):
   Computes node-to-edge scalar field gradients $\nabla T_{ij} = T_j - T_i$ to isolate thermal flux boundaries.

In [ ]:
from algebrax.analysis import gaussian_kernel, gradient
from algebrax.matrix.core import power
from algebrax.metrics import density, sparsity
from algebrax.semiring import ViterbiSemiring

## 1. Multi-Hop Max-Product Reliability (ViterbiSemiring)

We evaluate max-product path probability over lossy wireless links.

In [ ]:
link_probs = {0: {1: 0.90, 2: 0.70}, 1: {2: 0.85, 3: 0.95}, 2: {3: 0.60}, 3: {}}
viterbi = ViterbiSemiring()

rel_2step = power(link_probs, 2, semiring=viterbi)
rel_3step = power(link_probs, 3, semiring=viterbi)

best_p = max(rel_2step.get(0, {}).get(3, 0.0), rel_3step.get(0, {}).get(3, 0.0))
print(f'Optimal End-to-End Link Reliability P_max(0 -> 3): {best_p * 100:.2f}%')

## 2. Spatial RBF Gaussian Kernel & Sparsity Audit

We convert sensor distance metrics into spatial RBF similarities $K_{ij} = \exp(-d_{ij}^2 / (2 \sigma^2))$.

In [ ]:
distances = {0: {1: 1.5, 2: 3.0}, 1: {0: 1.5, 2: 1.0, 3: 4.0}, 2: {0: 3.0, 1: 1.0, 3: 2.0}, 3: {1: 4.0, 2: 2.0}}
rbf = gaussian_kernel(distances, sigma=2.0)

print('Spatial RBF Similarity Matrix:', rbf)
print(f'Network Density: {density(rbf, 16) * 100:.1f}%')

## 3. Discrete Thermal Field Gradients (gradient)

We compute node-to-edge scalar gradients $\nabla T_{ij} = T_j - T_i$.

In [ ]:
temp_field = {0: 22.0, 1: 45.0, 2: 48.0, 3: 23.0}
topology = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1, 3], 3: [1, 2]}

temp_grad = gradient(temp_field, topology)
print('Node-to-Edge Thermal Gradient grad(T)_ij:')
for u, row in temp_grad.items():
    print(f'  Node {u}: {row}')